In [1]:
!pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpy

In [7]:
import pandas as pd
import os
import re

# --- 1. PENGATURAN PENGGUNA (Silakan edit bagian ini) ---

# [WAJIB] Nama file input EXCEL Anda yang berisi koordinat DMS.
# Pastikan file ini berada di folder yang sama dengan skrip.
FILE_INPUT_EXCEL = "nage_deg.xlsx"

# (Opsional) Jika data Anda tidak berada di sheet pertama, tentukan nama sheet-nya di sini.
# Jika tidak diisi, skrip akan membaca sheet pertama secara default.
NAMA_SHEET = None

# [WAJIB] Nama file output CSV yang akan berisi koordinat Derajat Desimal.
FILE_OUTPUT_DESIMAL = "nage_des.csv"

# [WAJIB] Nama kolom di file input Anda yang berisi data DMS.
# Pastikan nama ini sama persis dengan yang ada di header file Excel Anda.
KOLOM_LATITUDE_DMS = "lat"
KOLOM_LONGITUDE_DMS = "lon"

# --- 2. FUNGSI KONVERSI (Telah diperbarui agar lebih tangguh) ---

def dms_ke_desimal(dms_string):
    """
    Mengubah string koordinat DMS (Derajat, Menit, Detik) menjadi Derajat Desimal.
    Fungsi ini sekarang bisa menangani format seperti "8° 52' 30\" S" dan "S8°52'15.2\"".
    """
    try:
        dms_string = str(dms_string).strip().upper()
        direction = None

        # Mendeteksi arah (N, S, E, W) di awal atau akhir string
        if dms_string.startswith(('N', 'S', 'E', 'W')):
            direction = dms_string[0]
            dms_string = dms_string[1:]
        elif dms_string.endswith(('N', 'S', 'E', 'W')):
            direction = dms_string[-1]
            dms_string = dms_string[:-1]

        # Menggunakan regex untuk menemukan semua angka (integer atau float) di dalam string
        numbers = re.findall(r"(\d+\.?\d*)", dms_string)

        if len(numbers) == 0:
            raise ValueError("Tidak ada angka yang ditemukan di dalam string DMS")

        # Menetapkan nilai derajat, menit, dan detik dari angka yang ditemukan
        derajat = float(numbers[0])
        menit = float(numbers[1]) if len(numbers) > 1 else 0
        detik = float(numbers[2]) if len(numbers) > 2 else 0

        # Menghitung nilai desimal
        desimal = derajat + (menit / 60) + (detik / 3600)

        # Menerapkan tanda negatif untuk Lintang Selatan (S) dan Bujur Barat (W)
        if direction in ['S', 'W']:
            desimal *= -1

        return desimal
    except (ValueError, IndexError, TypeError) as e:
        print(f"Gagal mengkonversi '{dms_string}': {e}. Mengembalikan None.")
        return None

# --- 3. PROSES UTAMA (Tidak perlu diedit) ---

# Memeriksa apakah file input ada
if not os.path.exists(FILE_INPUT_EXCEL):
    print(f"Error: File '{FILE_INPUT_EXCEL}' tidak ditemukan. Pastikan nama file sudah benar.")
else:
    print(f"Membaca file input: '{FILE_INPUT_EXCEL}'")
    try:
        # Membaca file Excel. Jika NAMA_SHEET tidak diisi, akan membaca sheet pertama.
        if NAMA_SHEET:
            df = pd.read_excel(FILE_INPUT_EXCEL, sheet_name=NAMA_SHEET)
        else:
            df = pd.read_excel(FILE_INPUT_EXCEL)

        print("Memulai proses konversi...")

        # Menerapkan fungsi konversi ke setiap baris di kolom Latitude dan Longitude
        df['lat_desimal'] = df[KOLOM_LATITUDE_DMS].apply(dms_ke_desimal)
        df['lon_desimal'] = df[KOLOM_LONGITUDE_DMS].apply(dms_ke_desimal)

        print("\nData setelah konversi (5 baris pertama):")
        print(df.head())

        # Menyimpan DataFrame yang sudah dikonversi ke file CSV baru
        df.to_csv(FILE_OUTPUT_DESIMAL, index=False)

        print(f"\nKonversi berhasil! Hasil disimpan di '{FILE_OUTPUT_DESIMAL}'")

    except FileNotFoundError:
        print(f"Error: File '{FILE_INPUT_EXCEL}' tidak ditemukan.")
    except KeyError as e:
        print(f"Error: Kolom {e} tidak ditemukan di file Excel. Periksa pengaturan 'KOLOM_LATITUDE_DMS' dan 'KOLOM_LONGITUDE_DMS'.")
    except Exception as e:
        print(f"Terjadi kesalahan tak terduga: {e}")
        

Membaca file input: 'nage_deg.xlsx'
Memulai proses konversi...
Gagal mengkonversi 'AN': Tidak ada angka yang ditemukan di dalam string DMS. Mengembalikan None.

Data setelah konversi (5 baris pertama):
   STA NO          lat            lon  lat_desimal  lon_desimal
0       1  S8°52′15.2″  E121°00′07.0″    -8.870889   121.001944
1       2  S8°52′29.4″  E121°00′04.7″    -8.874833   121.001306
2       3  S8°52′33.5″  E121°00′22.6″    -8.875972   121.006278
3       4  S8°52′52.1″  E121°00′51.0″    -8.881139   121.014167
4       5  S8°52′17.6″  E120°59′45.8″    -8.871556   120.996056

Konversi berhasil! Hasil disimpan di 'nage_des.csv'
